# Aula 7 — Validação e Otimização de Modelos
## *Treinar um modelo é fácil. Saber se ele presta é outra história.*

---

> **Data:** *(preencher)*  
> **Professor(a):** *(preencher)*

## Antes de começar: uma história

Um médico precisa diagnosticar se um paciente tem ou não uma doença grave.

Ele desenvolve um novo exame e testa em 100 pacientes. O exame acerta **95 de 100**. Parece incrível, né?

Mas aí você descobre um detalhe: **95 desses 100 pacientes não tinham a doença**.

O exame simplesmente dizia *"saudável"* para todo mundo — e acertava 95% das vezes **sem aprender absolutamente nada**.

Os 5 doentes de verdade? Todos passaram como saudáveis.

**Esse é o tipo de armadilha que mata projetos de Machine Learning — e às vezes, literalmente, pacientes.**

Nessa aula, você vai aprender a não cair nessa. Vamos entender de verdade como avaliar, validar e questionar um modelo.

---

## Setup — recriando o pipeline das Aulas 4 e 5

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error

# Dataset
df = sns.load_dataset('diamonds')

# Encoding
df_modelo = df.copy()
le = LabelEncoder()
for col in ['cut', 'color', 'clarity']:
    df_modelo[col] = le.fit_transform(df_modelo[col])

X = df_modelo.drop(columns=['price'])
y = df_modelo['price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

modelo = LinearRegression()
modelo.fit(X_train_sc, y_train)

y_pred_treino = modelo.predict(X_train_sc)
y_pred_teste  = modelo.predict(X_test_sc)

print("Pipeline recriado com sucesso!")
print(f"Dataset: {df.shape[0]:,} diamantes, {df.shape[1]} colunas")

---

# PARTE 1 — O que significa um modelo ser "bom"?

## 1.1. O problema com o MAE e R² sozinhos

Na Aula 4, calculamos MAE e R² e ficamos satisfeitos. Mas esses números têm um problema fundamental: **eles respondem "quanto o modelo errou", mas não respondem "por quê" nem "em quais situações".**

Pense assim: um aluno tirou 7 na prova. Isso é bom? Depende:
- Depende de como os outros foram.
- Depende de se ele estudou as questões certas ou teve sorte.
- Depende de se ele errou questões fáceis ou difíceis.
- Depende de se a nota de corte é 5 ou 9.

Com modelos é igual. Para saber se o modelo **realmente aprendeu**, precisamos de mais do que uma nota — precisamos entender o padrão dos acertos e erros.

Vamos construir esse entendimento passo a passo.

---

# PARTE 2 — Overfitting e Underfitting

## 2.1. O que são e por que existem

Todo modelo de Machine Learning tem uma tarefa: **encontrar um padrão nos dados de treino que se generalize para dados novos**.

Esse equilíbrio pode falhar de dois jeitos opostos:

---

### Underfitting — o modelo que não aprendeu nada útil

Acontece quando o modelo é **simples demais** para capturar a complexidade real dos dados.

**Analogia:** imagine que você vai aprender a reconhecer gatos. Sua regra é: *"se tem quatro patas, é gato"*. Simples demais — cachorro, vaca e cavalo também têm quatro patas. Você vai errar muito.

No ML: uma linha reta tentando representar uma relação que claramente tem curvatura.

**Sintoma:** erro alto tanto no treino quanto no teste.

---

### Overfitting — o modelo que decorou sem entender

Acontece quando o modelo é **complexo demais** e aprende até o ruído dos dados de treino — coisas que são acidentes, não padrões reais.

**Analogia:** um estudante que decorou as respostas de todos os exercícios do livro. Na prova, as questões são ligeiramente diferentes — e ele trava. Ele não entendeu o conceito, apenas memorizou os exemplos.

No ML: uma curva de grau altíssimo que passa exatamente por todos os pontos do treino, mas oscila de forma absurda entre eles.

**Sintoma:** erro baixo no treino, erro alto no teste. Grande diferença entre os dois.

---

### O ponto ideal — generalização

O objetivo é um modelo que capture o padrão real dos dados sem decorar os detalhes aleatórios. Na prática, isso significa:
- Erro de treino razoável (não zero — dado real tem ruído)
- Erro de teste próximo ao de treino

Vamos ver isso visualmente:

In [ ]:
# Dados sintéticos para a visualização


## 2.2. Como diagnosticar: treino vs teste

O diagnóstico mais direto é simples: **calcule a métrica separadamente no treino e no teste e compare**.

A lógica:
- O modelo viu os dados de treino durante o aprendizado. Esperamos que ele vá bem ali.
- Os dados de teste são novos. Se o modelo vai igualmente bem — ele generalizou.
- Se vai **muito melhor** no treino do que no teste — ele decorou (overfitting).
- Se vai **mal nos dois** — ele não aprendeu (underfitting).

In [ ]:
# Seu código aqui


---

# PARTE 3 — Curvas de Aprendizado

## 3.1. O que é e por que existe

A comparação treino vs teste te diz **o estado atual** do modelo. Mas ela não te diz **como chegou lá** — nem o que fazer para melhorar.

A **Curva de Aprendizado** responde uma pergunta diferente:

> *"Se eu der mais dados ao modelo, ele melhora? Ou já chegou no seu limite?"*

A ideia é treinar o mesmo modelo várias vezes, cada vez com uma quantidade diferente de dados, e plotar como o erro evolui.

**Como funciona na prática:**
- Começamos com poucos dados (ex: 5% do treino) → treinamos → medimos erro no treino e no teste
- Aumentamos para 10% → treinamos de novo → medimos
- Repetimos até usar 100% dos dados de treino

**O que esperamos ver:**

| Situação | Curva de treino | Curva de teste | Diagnóstico |
|---|---|---|---|
| Modelo saudável | Sobe levemente | Cai e estabiliza próxima ao treino | Tudo bem ✅ |
| Underfitting | Alta e plana | Alta e plana (próximas) | Modelo simples demais — mudar o modelo |
| Overfitting | Baixa | Alta e distante | Modelo complexo demais — mais dados ou regularização |
| Falta de dados | Caindo ainda | Caindo ainda | Coletar mais dados vai ajudar |

**Por que a curva de treino *sobe* com mais dados?**

Parece contra-intuitivo, mas faz sentido: com poucos dados, o modelo consegue quase "decorar" tudo — erro baixíssimo. Com mais dados, fica impossível decorar tudo, e o erro sobe até estabilizar no nível de dificuldade real do problema.

In [ ]:
# --- Gráfico 1: eixo Y do zero para mostrar proporção real ---

# --- Gráfico 2: zoom para ver a convergência ---

# Anotação


**Lendo o gráfico da esquerda:** a diferença entre as curvas parece mínima em escala real — as curvas convergem bem.

**Lendo o gráfico da direita (zoom):** conseguimos ver que as curvas se aproximam conforme aumentamos os dados, e estabilizam juntas. Esse é o comportamento de um modelo **sem overfitting**.

> **Dica importante:** a faixa sombreada ao redor de cada curva representa a variação entre as 5 dobras do cross-validation. Quanto menor essa faixa, mais estável é o modelo.

---

# PARTE 4 — Cross-Validation

## 4.1. O problema com o split único

Até agora, dividimos nossos dados uma única vez: 80% treino, 20% teste.

Parece razoável. Mas tem um problema escondido.

Imagine que você avalia um jogador de futebol assistindo a **um único jogo**. Se ele jogou bem naquele dia, você diz que é um bom jogador. Se jogou mal, descarta.

Mas e se ele estava machucado naquele dia? E se o adversário era fraquíssimo?

**Uma amostra só não é suficiente para uma conclusão confiável.**

Com modelos de ML, a mesma lógica se aplica:
- Talvez os 20% que foram para o teste foram justamente os casos mais fáceis → modelo parece melhor do que é
- Ou foram os casos mais difíceis → modelo parece pior do que é
- O resultado depende do acaso da divisão

## 4.2. K-Fold Cross-Validation — a solução

A ideia é testar o modelo em **todas as partes dos dados**, não só numa.

No **K-Fold com K=5**, dividimos os dados em 5 partes iguais (chamadas *folds*). Depois rodamos 5 experimentos:

```
Rodada 1:  [TESTE  | treino | treino | treino | treino]  → calcula MAE_1
Rodada 2:  [treino | TESTE  | treino | treino | treino]  → calcula MAE_2
Rodada 3:  [treino | treino | TESTE  | treino | treino]  → calcula MAE_3
Rodada 4:  [treino | treino | treino | TESTE  | treino]  → calcula MAE_4
Rodada 5:  [treino | treino | treino | treino | TESTE ]  → calcula MAE_5

Resultado final = média(MAE_1, MAE_2, MAE_3, MAE_4, MAE_5)
```

Cada dado é usado como teste exatamente **uma vez**. O resultado é muito mais robusto.

**Por que não usar sempre K=100?**

Quanto maior o K:
- ✅ Estimativa mais precisa do desempenho
- ❌ Mais demorado computacionalmente
- ❌ Cada fold de teste fica menor (menos representativo)

O valor **K=5 ou K=10** é o padrão da indústria — equilíbrio entre precisão e custo computacional.

In [ ]:
# Testando diferentes valores de K


In [ ]:
# Visualizando a variação entre folds para K=5

# Gráfico 1: MAE por fold

# Gráfico 2: Comparação split unico vs cross-val


**O que o desvio padrão entre os folds nos diz?**

Se os 5 folds deram MAEs muito diferentes entre si (ex: 600, 900, 750, 1100, 680), o modelo é instável — o desempenho depende muito de quais dados ele vê.

Se os 5 folds deram MAEs similares (ex: 840, 860, 855, 870, 848), o modelo é estável e confiável.

> **Regra prática:** quando comparar dois modelos, prefira o que tem **menor média de MAE** no cross-validation. Se a diferença de média for pequena, prefira o de **menor desvio padrão** (mais previsível).

---

# PARTE 5 — Baseline

## 5.1. Comparado com o quê?

Seu modelo tem MAE de $860. É bom?

Depende do que você compara. Se um modelo que **não usa nenhuma variável** e simplesmente chuta a média do preço já tem MAE de $870, então seu modelo sofisticado de ML economizou apenas $10 de erro. Valeu toda a complexidade?

O **Baseline** é o nível mínimo que qualquer modelo decente precisa superar. É o *piso* de qualidade.

### Por que o baseline é obrigatório?

Sem baseline, você não sabe se o seu modelo é bom em termos absolutos ou apenas "menos ruim que o nada".

**Analogia:** você desenvolveu um novo remédio contra dor de cabeça e ele funciona em 70% dos casos. Ótimo!

Mas aí você descobre que placebo (pílula de açúcar) também funciona em 65% dos casos. Seu remédio ainda é útil? Sim, mas apenas marginalmente — e custa muito mais.

### Tipos de baseline para regressão:
- **Média:** chuta sempre a média do alvo → o mais comum
- **Mediana:** mais robusto com distribuições assimétricas
- **Modelo simples:** regressão com apenas uma variável, sem engenharia de features

In [ ]:
# Baseline 1: sempre chuta a média

# Baseline 2: sempre chuta a mediana


---

# PARTE 6 — Métricas de Erro em Detalhe

## 6.1. MAE — Mean Absolute Error

**O que é:** a média dos erros absolutos de cada previsão.

**Como calcular (em palavras):**
1. Para cada diamante, calcule: `|preço real - preço previsto|`
2. Tire a média de todos esses erros

**Interpretação:** *"em média, o modelo erra por $X"*

**Vantagem:** fácil de entender. O erro está na mesma unidade da variável alvo.

**Limitação:** trata todos os erros igualmente — um erro de $100 e um erro de $10.000 têm o mesmo peso proporcional.

---

## 6.2. RMSE — Root Mean Squared Error

**O que é:** similar ao MAE, mas eleva os erros ao quadrado antes de calcular a média.

**Como calcular (em palavras):**
1. Para cada diamante: `(preço real - preço previsto)²`
2. Tire a média desses quadrados
3. Tire a raiz quadrada do resultado

**Por que elevar ao quadrado?** Erros grandes ficam ainda maiores. Um erro de $200 vira 40.000. Um erro de $2.000 vira 4.000.000. O quadrado **pune desproporcionalmente os erros grandes**.

**Interpretação:** *"o modelo tem um erro típico de $X, sendo que erros grandes puxam esse número para cima"*

**Quando usar:** quando erros grandes são desastrosos. No contexto médico, um erro grande pode matar alguém — você quer saber sobre isso. Em finanças, uma previsão muito errada pode causar prejuízo enorme.

**Sinal de alerta:** se RMSE >> MAE, existem erros pontuais muito grandes. Investigue quais casos são esses.

---

## 6.3. MAPE — Mean Absolute Percentage Error

**O que é:** o erro em percentual do valor real.

**Como calcular (em palavras):**
1. Para cada diamante: `|preço real - preço previsto| / preço real`
2. Multiplique por 100 para ter percentual
3. Tire a média

**Por que existe:** $500 de erro em um diamante de $600 é catastrófico (83% de erro). $500 de erro em um diamante de $15.000 é irrelevante (3% de erro). O MAE não distingue esses casos — o MAPE sim.

**Interpretação:** *"em média, o modelo erra X% do valor real"*

**Quando usar:** quando você precisa comunicar o resultado para pessoas de negócio. "Erramos em média 12%" é muito mais intuitivo do que "MAE = $740".

**Limitação:** não funciona quando o valor real é zero (divisão por zero) e é sensível a valores muito pequenos.

---

## 6.4. R² — Coeficiente de Determinação

**O que é:** a proporção da variação do alvo que o modelo consegue explicar.

**Interpretação:** R² = 0.91 significa que o modelo explica 91% da variação nos preços. Os outros 9% são variação que o modelo não capturou.

**Como pensar:** imagine que o preço dos diamantes varia muito — de $300 a $18.000. Um modelo perfeito explicaria toda essa variação. O R² mede o quanto do "espalhamento" dos dados o modelo consegue justificar com as variáveis que tem.

**Cuidado:** R² alto não significa modelo bom necessariamente:
- Com poucos dados, é fácil ter R² alto por overfitting
- R² não captura padrões nos resíduos
- R² pode ser alto mesmo com um modelo que tem viés sistemático

In [ ]:
# --- Gráfico 1: Comparação MAE vs RMSE ---

# --- Gráfico 2: Distribuição dos erros absolutos ---

# --- Gráfico 3: Tabela resumo ---


---

# PARTE 7 — Matriz de Confusão

## 7.1. Por que precisamos disso se estamos fazendo regressão?

Até aqui trabalhamos com **regressão** — prever um número (preço).

Mas muitos problemas do mundo real são de **classificação** — prever uma categoria:
- Email: spam ou não spam?
- Transação: fraude ou legítima?
- Exame médico: positivo ou negativo?
- Cliente: vai cancelar o plano ou não?

Para problemas de classificação, as métricas de erro (MAE, RMSE) não fazem sentido — você não calcula a raiz quadrada de "spam" menos "não spam".

Precisamos de outras ferramentas. A principal é a **Matriz de Confusão**.

## 7.2. O que é a Matriz de Confusão

A Matriz de Confusão organiza os acertos e erros do modelo em uma tabela, separando **quais tipos de erro** aconteceram.

Vamos usar um exemplo concreto: **detectar diamantes caros** (acima de $5.000).

O modelo classifica cada diamante como "caro" ou "barato". Existem 4 resultados possíveis:

```
                        REALIDADE
                   Caro      |    Barato
              ┌─────────────────────────────┐
    PREVISTO  │  Verdadeiro  │   Falso      │
    Caro      │  Positivo    │   Positivo   │  ← Previu CARO
              │    (VP)      │     (FP)     │
              ├─────────────────────────────┤
    PREVISTO  │  Falso       │  Verdadeiro  │
    Barato    │  Negativo    │  Negativo    │  ← Previu BARATO
              │    (FN)      │    (VN)      │
              └─────────────────────────────┘
```

**Traduzindo:**
- **VP (Verdadeiro Positivo):** previu caro, era caro. ✅ Acertou!
- **VN (Verdadeiro Negativo):** previu barato, era barato. ✅ Acertou!
- **FP (Falso Positivo):** previu caro, mas era barato. ❌ Alarme falso.
- **FN (Falso Negativo):** previu barato, mas era caro. ❌ Deixou escapar.

## 7.3. Por que separar FP e FN importa tanto?

**Depende do problema, os erros têm custos diferentes!**

| Contexto | Falso Positivo (FP) | Falso Negativo (FN) | Qual é pior? |
|---|---|---|---|
| Detecção de câncer | Diagnóstico positivo falso → exame desnecessário | Não detectar câncer real → paciente morre sem tratamento | FN é catastrófico |
| Filtro de spam | Email legítimo vai para spam → frustração | Spam passa → inconveniente | FP é mais grave |
| Detecção de fraude | Transação legítima bloqueada → cliente irritado | Fraude não detectada → prejuízo financeiro | Depende do custo |

A acurácia sozinha não captura essa diferença. A Matriz de Confusão sim.

In [ ]:
# Criando problema de classificacao: diamante caro (>= 5000) ou barato?

# Treinando classificador

# Calculando a Matriz de Confusao

# Plotando

# Matriz de confusao visual

# Proporcoes


## 7.4. Métricas derivadas da Matriz de Confusão

A partir dos 4 valores da matriz, calculamos métricas mais específicas:

---

### Acurácia
**O que é:** proporção de previsões corretas sobre o total.

```
Acurácia = (VP + VN) / (VP + VN + FP + FN)
```

**Limitação crítica:** pode ser enganosa em classes desbalanceadas. Lembra do médico da introdução? 95% de acurácia dizendo "saudável" para todo mundo.

---

### Precisão (Precision)
**O que é:** dos que o modelo disse "positivo", quantos eram realmente positivos?

```
Precisão = VP / (VP + FP)
```

**Em palavras:** *"quando o modelo afirma que é caro, com qual frequência ele está certo?"*

**Quando importa:** quando um Falso Positivo é caro. Ex: bloquear uma transação legítima como fraude → cliente irritado.

---

### Recall (Sensibilidade)
**O que é:** dos que eram realmente positivos, quantos o modelo encontrou?

```
Recall = VP / (VP + FN)
```

**Em palavras:** *"de todos os diamantes caros, quantos o modelo identificou corretamente?"*

**Quando importa:** quando um Falso Negativo é caro. Ex: não detectar um câncer real → tragédia.

---

### F1-Score
**O que é:** a média harmônica entre Precisão e Recall — um número único que equilibra os dois.

```
F1 = 2 × (Precisão × Recall) / (Precisão + Recall)
```

**Quando usar:** quando você quer uma métrica única e as duas classes importam igualmente.

> **A tensão Precisão vs Recall:** existe um trade-off natural. Se você aumenta o limiar para dizer "positivo" (exige mais certeza), a Precisão sobe (erra menos alarmes falsos) mas o Recall cai (deixa escapar mais casos reais). É impossível maximizar os dois ao mesmo tempo — você precisa decidir qual erro é mais aceitável no seu contexto.

In [ ]:
# Relatório completo do sklearn


In [ ]:
# Visualizando o trade-off Precisão vs Recall

# Curva Precisao vs Recall

# Precisão e Recall por limiar


---

# PARTE 8 — Resíduos: o modelo erra de forma aleatória?

## 8.1. O que é um resíduo e por que analisar

**Resíduo = Valor Real − Valor Previsto**

É o erro individual de cada previsão. Se o diamante custava $5.000 e o modelo previu $4.300, o resíduo é $700.

**Por que olhar para os resíduos individualmente, se já temos o MAE?**

O MAE te diz *quanto* o modelo erra em média. Os resíduos te dizem *como* o modelo erra — e isso revela se existe algum **padrão nos erros**.

**Um modelo bom erra de forma aleatória.** Se os erros têm padrão, significa que o modelo deixou de capturar algo importante.

**Analogia do arqueiro:** um arqueiro treinado errou o alvo 10 vezes. Se as flechas ficaram espalhadas aleatoriamente em volta do centro, ele está treinando bem — o erro é ruído natural. Mas se todas as flechas foram para a esquerda, ele tem um problema sistemático: talvez a mira esteja descalibrada, talvez o vento, talvez ele esteja doente. Esse padrão precisa ser corrigido.

**O que procurar nos gráficos de resíduo:**

| O que você vê | O que significa |
|---|---|
| Nuvem aleatória de pontos ao redor do zero | ✅ Modelo saudável, erros são ruído |
| Resíduos crescem conforme o valor previsto cresce | ⚠️ Heterocedasticidade — erro não é constante |
| Distribuição assimétrica (mais erros para um lado) | ⚠️ Viés sistemático — modelo superestima ou subestima |
| Formato de curva nos resíduos vs previsto | ⚠️ Relação não-linear não capturada |

In [ ]:
# --- Gráfico 1: Residuos vs Valores Previstos ---

# --- Gráfico 2: Distribuição dos resíduos ---

# --- Gráfico 3: Residuos vs Valor Real ---

# --- Gráfico 4: Estatísticas dos resíduos ---


**Conclusão da análise de resíduos:**

O modelo tem um problema claro: para diamantes caros (acima de ~$10.000), os resíduos ficam muito maiores e mais espalhados. Isso indica que a relação entre as variáveis e o preço **não é perfeitamente linear** para valores altos.

Isso não torna o modelo inútil — mas nos diz que para previsões de diamantes premium, precisaríamos de um modelo mais sofisticado.

---

# PARTE 9 — Teste de Hipótese e P-value

## 9.1. O problema que o teste de hipótese resolve

Você olha para os dados e percebe que diamantes com corte "Ideal" custam em média $200 mais que diamantes com corte "Premium".

Isso é real? Ou é só variação aleatória nos dados que você coletou?

Se você tivesse coletado outros 54.000 diamantes diferentes, essa diferença de $200 ainda apareceria? Ou sumia?

**O Teste de Hipótese é o método formal para responder essa pergunta.**

## 9.2. A lógica do Teste de Hipótese

O raciocínio funciona assim:

1. **Você parte do princípio que nada especial está acontecendo** — que a diferença que você observou é puro acaso. Isso se chama **Hipótese Nula (H₀)**.

2. **Você calcula: se a hipótese nula fosse verdadeira, qual a probabilidade de observar o que observei?** Essa probabilidade é o **p-value**.

3. **Se o p-value for muito baixo** (convencionalmente < 5%), significa que os dados que você viu são muito improváveis de acontecer por acaso. Logo, a hipótese nula provavelmente está errada — há algo real acontecendo.

## 9.3. A Hipótese Nula

A **Hipótese Nula (H₀)** é sempre a afirmação de que *não existe efeito*, *não existe diferença*, *não existe relação*.

Exemplos:
- "O tipo de corte **não** afeta o preço do diamante" → H₀
- "O coeficiente do quilate no modelo é **zero**" → H₀
- "O novo remédio **não** é melhor que o placebo" → H₀

A **Hipótese Alternativa (H₁)** é o oposto — que sim, existe efeito.

Você nunca prova que H₁ é verdadeira. Você apenas coleta evidências contra H₀. Se a evidência é forte o suficiente, você **rejeita a hipótese nula**.

## 9.4. O P-value em detalhe

**Definição formal:** o p-value é a probabilidade de observar um resultado **tão extremo quanto o observado**, assumindo que a hipótese nula é verdadeira.

**Em palavras simples:** *"se não houvesse efeito real, qual a chance de os dados terem ficado assim por coincidência?"*

**A analogia da moeda:**

Você suspeita que uma moeda é viciada. Joga 10 vezes e cai cara 8 vezes.

- Hipótese Nula: *"a moeda é honesta"*
- Pergunta: *"se a moeda fosse honesta, qual a chance de cair cara 8 ou mais vezes em 10 lançamentos?"*
- Resposta: ~5.5% → p-value ≈ 0.055

Isso é ligeiramente acima de 5% — evidência fraca contra a moeda honesta.

Agora jogue 100 vezes e caia cara 70. O p-value seria minúsculo → evidência fortíssima de que a moeda é viciada.

**O limiar de 5% (p < 0.05):**

É uma convenção histórica da ciência. Significa: *"só vou concluir que existe um efeito real se a chance de ser coincidência for menor que 5%"*.

Não é um número mágico — em medicina usa-se 1%, em física usa-se 0.0001%. Depende do contexto e do custo de errar.

**O que p-value NÃO é:**
- Não é a probabilidade de H₀ ser verdadeira
- Não mede o tamanho do efeito (algo pode ser significante estatisticamente mas irrelevante na prática)
- Um p-value alto não prova que H₀ é verdadeira — só que não temos evidência suficiente contra ela

## 9.5. Aplicando na Regressão: p-value dos coeficientes

Na regressão linear, cada variável tem um coeficiente. O p-value de cada coeficiente responde:

> *"Se esse coeficiente fosse zero na realidade (variável não tem efeito), qual a chance de observarmos um coeficiente tão grande quanto o que calculamos?"*

- p < 0.05 → evidência de que essa variável tem efeito real no preço ✅
- p ≥ 0.05 → não temos evidência suficiente — pode ser coincidência ❌

In [ ]:
# statsmodels exige adicionar a constante (intercepto) manualmente

# Ajustando o modelo de regressão com estatísticas completas


In [ ]:
# Extraindo de forma legível


In [ ]:
# Visualização dos coeficientes com intervalo de confiança

# Gráfico 1: Coeficientes com IC

# Anotando p-values

# Gráfico 2: Explicação do Intervalo de Confiança


## 9.6. Correlação vs Significância — a diferença que muda tudo

Essa é uma das confusões mais comuns em Ciência de Dados:

**Correlação** pergunta: *"X e Y estão associados nos dados?"*

**Significância** pergunta: *"Essa associação é real ou pode ser coincidência?"*

São perguntas diferentes, e os resultados podem divergir:

| Situação | Correlação | Significância | O que fazer |
|---|---|---|---|
| Alta correlação, p < 0.05 | Alta | Sim | Manter a variável ✅ |
| Alta correlação, p ≥ 0.05 | Alta | Não | Investigar — dados insuficientes? |
| Baixa correlação, p < 0.05 | Baixa | Sim | Efeito real mas pequeno — depende do contexto |
| Baixa correlação, p ≥ 0.05 | Baixa | Não | Remover a variável |

Com 54.000 diamantes, quase tudo vai ser significante estatisticamente (grande volume de dados reduz o p-value). **Nesse caso, o tamanho do coeficiente importa mais do que a significância.**

> **Regra de ouro:** em datasets grandes, p-value baixo é quase garantido. O que importa é o **tamanho do efeito** — o coeficiente é grande o suficiente para ser relevante na prática?

---

# PARTE 10 — Dashboard Final de Avaliação

In [ ]:
# 1. Previsoes vs Real

# 2. Residuos vs Previsto

# 3. Distribuicao residuos

# 4. Treino vs Teste

# 5. Baseline

# 6. Cross-Validation

# 7. Matriz de confusao

# 8. Metricas resumo

# 9. Coeficientes


---

# Checklist de Avaliação — Leve para a vida

```
ANTES DE DECLARAR QUE SEU MODELO E BOM:

[ ] 1. Comparei com o BASELINE?
       O modelo e melhor do que chutar a media?

[ ] 2. Comparei TREINO vs TESTE?
       Erro parecido nos dois? Ou overfitting?

[ ] 3. Usei CROSS-VALIDATION?
       O resultado e estavel entre os folds?

[ ] 4. Analisei as METRICAS certas para o contexto?
       - Erros grandes sao graves? -> olha o RMSE
       - Precisa comunicar para nao-tecnicos? -> usa MAPE
       - Classificacao com classes desbalanceadas? -> nao usa so acuracia

[ ] 5. Olhei a MATRIZ DE CONFUSAO (se for classificacao)?
       Que tipo de erro e mais grave: FP ou FN?

[ ] 6. Analisei os RESIDUOS?
       Ha algum padrao nos erros?

[ ] 7. Verifiquei os P-VALUES?
       Quais variaveis tem efeito real?

[ ] 8. O modelo FAZ SENTIDO?
       Os coeficientes estao na direcao esperada?
       Ou alguma coisa absurda esta acontecendo?
```

---

# Exercícios

---

## 🟢 Fácil

**Exercício 1 — Diagnóstico treino vs teste**

Simule overfitting: treine o modelo com apenas **100 diamantes** e avalie no teste completo.
- Qual o MAE no treino? E no teste?
- A diferença é maior do que com o conjunto completo de treino?
- Por que isso acontece?

In [ ]:
# Exercicio 1


**Exercício 2 — MAPE vs MAE**

Calcule o MAPE do baseline (media) e da regressao linear.
- A melhoria percentual é maior no MAE ou no MAPE?
- O que isso indica sobre onde o modelo melhora mais: em diamantes baratos ou caros?

In [ ]:
# Exercicio 2


---

## 🟡 Médio

**Exercício 3 — Cross-validation e estabilidade**

Rode cross-validation com K=5 e K=10.
- O MAE médio muda entre os dois?
- O desvio padrão muda? Em qual direção?
- Com base nisso, existe razão para usar K=10 nesse dataset? Justifique.

In [ ]:
# Exercicio 3


**Exercício 4 — Matriz de Confusão com limiar diferente**

No classificador de diamantes caros, o limiar padrão é 0.5 (se a probabilidade > 50%, classifica como caro).

Mude o limiar para **0.3** e depois para **0.7**:
- Como muda a Precisao e o Recall em cada caso?
- Se você fosse um comprador que quer ter certeza de nao pagar caro por algo barato, qual limiar preferiria? E se fosse um vendedor querendo nao deixar escapar nenhum diamante caro?

*Dica: use `clf.predict_proba(X_test_sc)[:, 1] > 0.3` para classificar com limiar 0.3*

In [ ]:
# Exercicio 4


---

## 🔴 Difícil

**Exercício 5 — P-values e seleção de variáveis**

Usando o `modelo_sm` (statsmodels):

1. Identifique todas as variáveis com p-value >= 0.05.
2. Remova essas variáveis e retreine o modelo com statsmodels.
3. Compare R², MAE (no teste via sklearn), e os novos p-values.
4. Reflita: remover variáveis não-significantes sempre melhora o modelo? Por quê?

*Dica: lembre de remover a variável tanto de X_train_sc quanto de X_test_sc — use índices de coluna.*

In [ ]:
# Exercicio 5


**Exercício 6 — Análise completa com transformação log**

Uma limitação do modelo é que os residuos crescem para diamantes caros. Uma solução clássica é aplicar log no alvo antes de treinar.

Faça o pipeline completo com `y_log = np.log(y)` como alvo:

1. Treine o modelo com y_log
2. Converta as previsoes de volta com `np.exp(y_pred_log)`
3. Calcule MAE, RMSE, MAPE e R² originais
4. Plote os residuos (usando os valores convertidos de volta)
5. Rode cross-validation
6. Compare tudo com o modelo sem log

Pergunta final: o modelo com log resolve o problema dos residuos crescentes? Por que faz sentido aplicar log no preco de diamantes?

In [ ]:
# Exercicio 6


---

## Resumo Final

| Conceito | O que é | Para que serve |
|---|---|---|
| **Overfitting** | Modelo decorou os dados de treino | Diagnosticado comparando treino vs teste |
| **Underfitting** | Modelo simples demais | Ambos os erros sao altos |
| **Learning Curve** | Erro vs quantidade de dados | Ver se mais dados ajudam |
| **Cross-Validation** | Testar em todas as partes dos dados | Avaliacao robusta e confiavel |
| **Baseline** | Modelo idiota de referencia | Piso minimo de qualidade |
| **MAE** | Erro medio absoluto | Interpretacao direta |
| **RMSE** | Penaliza erros grandes | Quando grandes erros sao criticos |
| **MAPE** | Erro percentual | Comunicacao para nao-tecnicos |
| **Matriz de Confusao** | Organiza tipos de erros | Classificacao — entender FP e FN |
| **Precisao** | Dos que previ positivo, acertei? | Minimizar alarmes falsos |
| **Recall** | Dos positivos reais, achei? | Minimizar casos nao detectados |
| **F1-Score** | Equilibrio P e R | Quando as duas classes importam |
| **Residuos** | Padrao nos erros | Detectar problemas estruturais |
| **Hipotese Nula** | Assume que nao existe efeito | Base do teste de hipotese |
| **P-value** | Prob. do efeito ser coincidencia | p < 0.05 = efeito real |
| **statsmodels** | Regressao com estatisticas | Coeficientes + p-values + IC |

> **A frase que resume a aula:** *Um modelo nao e bom porque o R² e alto. E bom quando sobrevive a todas as perguntas desta tabela.*